In [2]:
# Import the tools used for data analysis and recommendations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

### Operations Director focus

This analysis supports the Operations Director by identifying customer demand, popular products and purchasing patterns. These findings can help with stock planning, product availability and store operations.

In [3]:
# Load the customer, product and transaction datasets

customers_df = pd.read_csv("csv/Customers_members.csv")
products_df = pd.read_csv("csv/Products_overview.csv")
transactions_df = pd.read_csv("csv/Transactions_history.csv")

In [4]:
# Show the first five customer records

customers_df.head()

,CustomerID,AgeGroup,Region,LoyaltyLevel,MembershipYears
0,1,36-50,East,Bronze,9
1,2,26-35,East,Bronze,4
2,3,18-25,East,Silver,10
3,4,36-50,East,Bronze,14
4,5,26-35,North,Bronze,7


In [5]:
# Show the first five product records

products_df.head()

,ProductID,Category,Brand,Price,Rating,PopularityScore
0,1,Electronics,TechMax,474.45,4.54,3
1,2,Gaming,SmartBuy,285.11,3.16,2
2,3,books,HomePro,370.79,3.01,1
3,4,Home,TechMax,416.38,3.38,1
4,5,Books,TechMax,62.37,3.42,1


In [6]:
# Show the first five transaction records

transactions_df.head()

,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
0,1,82,3,162,104.01
1,2,7,4,276,387.48
2,3,170,4,215,1507.40
3,3,9,1,251,341.40
4,4,64,3,354,1250.40


In [7]:
# Check the number of rows and columns in each dataset

print("customers_df shape:", customers_df.shape)
print("products_df shape:", products_df.shape)
print("transactions_df shape:", transactions_df.shape)

customers_df shape: (1000, 5)
products_df shape: (200, 6)
transactions_df shape: (19166, 5)


In [8]:
# Check the data type of every column

display(customers_df.dtypes.rename("Data Type").to_frame())
display(products_df.dtypes.rename("Data Type").to_frame())
display(transactions_df.dtypes.rename("Data Type").to_frame())

,Data Type
CustomerID,int64
AgeGroup,str
Region,str
LoyaltyLevel,str
MembershipYears,int64


,Data Type
ProductID,int64
Category,str
Brand,str
Price,float64
Rating,float64
PopularityScore,int64


,Data Type
CustomerID,int64
ProductID,int64
PurchaseCount,int64
LastPurchaseDays,int64
Spend,float64


In [9]:
# Check for missing values in the customer dataset

customers_df.isnull().sum().rename("Missing Values").to_frame()

,Missing Values
CustomerID,0
AgeGroup,0
Region,0
LoyaltyLevel,0
MembershipYears,0


In [10]:

# Check for missing values in the product dataset

products_df.isnull().sum().rename("Missing Values").to_frame()

,Missing Values
ProductID,0
Category,0
Brand,0
Price,0
Rating,15
PopularityScore,0


In [11]:
# Check for missing values in the transaction dataset

transactions_df.isnull().sum().rename("Missing Values").to_frame()

,Missing Values
CustomerID,0
ProductID,0
PurchaseCount,0
LastPurchaseDays,0
Spend,0


In [12]:
# Count exact duplicate rows in the customer dataset

duplicate_count = customers_df.duplicated().sum()
print("Number of duplicate rows in customers_df:", duplicate_count)

Number of duplicate rows in customers_df: 0


In [13]:
# Count exact duplicate rows in the product dataset

duplicate_count = products_df.duplicated().sum()
print("Number of duplicate rows in products_df:", duplicate_count)

Number of duplicate rows in products_df: 0


In [14]:
# Count exact duplicate rows in the transaction dataset

duplicate_count = transactions_df.duplicated().sum()
print("Number of duplicate rows in transactions_df:", duplicate_count)

Number of duplicate rows in transactions_df: 384


In [15]:
# Display examples of duplicate transaction rows

transactions_df[transactions_df.duplicated(keep=False)].head(10)

,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
4,4,64,3,354,1250.40
22,15,30,3,28,273.75
108,53,89,1,182,139.10
134,54,169,3,344,497.67
142,55,8,2,153,683.16
201,58,119,1,235,304.19
205,58,22,4,201,581.44
231,60,102,2,313,968.72
482,74,137,1,304,458.85
506,75,84,4,214,1042.20


In [16]:
# Count repeated customer-product pairs; repeat purchases can be valid

duplicate_count = transactions_df.duplicated(
    subset=["CustomerID", "ProductID"]
).sum()

print("Number of duplicate CustomerID and ProductID combinations:", duplicate_count)

Number of duplicate CustomerID and ProductID combinations: 3474


In [17]:
# Remove exact duplicate transaction records before integration

duplicate_transaction_rows = transactions_df.duplicated().sum()
transactions_df = transactions_df.drop_duplicates().reset_index(drop=True)

print(f"Removed {duplicate_transaction_rows} exact duplicate transaction rows.")
print(f"Remaining transaction rows: {len(transactions_df)}")

Removed 384 exact duplicate transaction rows.
Remaining transaction rows: 18782


In [18]:
# Standardise product categories before merging and analysis

products_df["Category"] = products_df["Category"].str.strip().str.title()

# Fill missing ratings with the median rating

products_df["Rating"] = products_df["Rating"].fillna(products_df["Rating"].median())

In [19]:
# Combine transaction data with customer and product information

merged_df = (
    transactions_df
    .merge(customers_df, on="CustomerID", how="left")
    .merge(products_df, on="ProductID", how="left")
)

print("merged_df shape:", merged_df.shape)
display(merged_df.head())

merged_df shape: (18782, 14)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend,AgeGroup,Region,LoyaltyLevel,MembershipYears,Category,Brand,Price,Rating,PopularityScore
0,1,82,3,162,104.01,36-50,East,Bronze,9,Books,HomePro,34.67,3.56,5
1,2,7,4,276,387.48,26-35,East,Bronze,4,Home,SmartBuy,96.87,3.79,49
2,3,170,4,215,1507.40,18-25,East,Silver,10,Kitchen,FitLife,376.85,2.78,4
3,3,9,1,251,341.40,18-25,East,Silver,10,Gaming,SmartBuy,341.40,4.23,3
4,4,64,3,354,1250.40,36-50,East,Bronze,14,Gaming,TechMax,416.80,3.59,1


In [20]:
missing_after_merge = merged_df.isnull().sum().rename("Missing Values").to_frame()
display(missing_after_merge)

,Missing Values
CustomerID,0
ProductID,0
PurchaseCount,0
LastPurchaseDays,0
Spend,0
AgeGroup,0
Region,0
LoyaltyLevel,0
MembershipYears,0
Category,0


In [21]:
# Find the ten customers who spent the most

top_10_customers = (
    merged_df.groupby("CustomerID", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalSpend"})
    .sort_values("TotalSpend", ascending=False)
    .head(10)
)

display(top_10_customers.style.hide(axis="index"))

CustomerID,TotalSpend
395,25504.830000
751,23084.090000
298,22883.980000
454,22750.970000
57,22713.060000
546,22399.850000
958,22132.060000
931,22050.150000
177,21954.580000
879,21698.920000


In [22]:
# Find the ten customers with the most purchases

top_10_customers_by_frequency = (
    merged_df.groupby("CustomerID", as_index=False)["PurchaseCount"]
    .sum()
    .rename(columns={"PurchaseCount": "TotalPurchases"})
    .sort_values("TotalPurchases", ascending=False)
    .head(10)
)

display(top_10_customers_by_frequency.style.hide(axis="index"))

CustomerID,TotalPurchases
879,81
812,81
550,81
664,80
480,80
104,79
831,79
912,79
656,79
395,79


In [23]:
# Calculate total spending, purchase count and average spend per purchase

customer_value_analysis = (
    merged_df.groupby("CustomerID", as_index=False)
    .agg(
        TotalSpend=("Spend", "sum"),
        TotalPurchases=("PurchaseCount", "sum")
    )
)

customer_value_analysis["AverageSpendPerPurchase"] = (
    customer_value_analysis["TotalSpend"]
    / customer_value_analysis["TotalPurchases"]
)

top_10_customer_value = (
    customer_value_analysis
    .sort_values("TotalSpend", ascending=False)
    .head(10)
)

display(
    top_10_customer_value.style
    .hide(axis="index")
    .format({
        "TotalSpend": "{:.2f}",
        "AverageSpendPerPurchase": "{:.2f}"
    })
)

CustomerID,TotalSpend,TotalPurchases,AverageSpendPerPurchase
395,25504.83,79,322.85
751,23084.09,72,320.61
298,22883.98,76,301.11
454,22750.97,76,299.35
57,22713.06,71,319.90
546,22399.85,79,283.54
958,22132.06,76,291.21
931,22050.15,76,290.13
177,21954.58,74,296.68
879,21698.92,81,267.89


In [24]:
# Find the ten products that generated the most revenue

top_10_products = (
    merged_df.groupby(["ProductID", "Category", "Brand"], as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
    .head(10)
)

display(
    top_10_products.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

ProductID,Category,Brand,TotalRevenue
102,Books,FitLife,1718993.64
7,Home,SmartBuy,476891.01
41,Home,HomePro,432435.78
134,Kitchen,SmartBuy,405060.82
8,Home,HomePro,234323.88
184,Fitness,SmartBuy,189363.83
45,Kitchen,NextGen,185750.73
65,Kitchen,TechMax,184635.64
81,Home,FitLife,178441.35
137,Electronics,NextGen,169315.65


In [25]:
# Compare total revenue across product categories

revenue_by_category = (
    merged_df.groupby("Category", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
)

display(
    revenue_by_category.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

Category,TotalRevenue
Books,2739415.70
Home,2291980.39
Electronics,1765677.88
Kitchen,1765394.06
Fitness,1545305.18
Gaming,1380970.91


In [26]:
# Display the product brands in the dataset

display(
    pd.DataFrame({"Brand": sorted(products_df["Brand"].dropna().unique())})
    .style.hide(axis="index")
)

Brand
FitLife
HomePro
NextGen
PrimeGoods
SmartBuy
TechMax


In [27]:
# Compare total revenue across product brands

revenue_by_brand = (
    merged_df.groupby("Brand", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
)

display(
    revenue_by_brand.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

Brand,TotalRevenue
FitLife,3283248.33
SmartBuy,2576635.12
HomePro,1862715.87
TechMax,1600803.26
NextGen,1272086.37
PrimeGoods,893255.17


In [28]:
# Compare spending by region and product category

region_category_spend = merged_df.pivot_table(
    index="Region",
    columns="Category",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

display(region_category_spend.style.format("{:.2f}"))

Category,Books,Electronics,Fitness,Gaming,Home,Kitchen
Region,,,,,,
East,677364.69,435323.31,379487.13,309305.85,575780.89,442294.35
North,654351.93,429234.75,390452.80,340786.69,545772.68,444784.51
South,679764.81,434826.70,352989.80,348171.49,590918.93,455267.93
West,727934.27,466293.12,422375.45,382706.88,579507.89,423047.27


In [29]:
# Compare spending by loyalty level and region

loyalty_region_spend = merged_df.pivot_table(
    index="LoyaltyLevel",
    columns="Region",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

display(loyalty_region_spend.style.format("{:.2f}"))

Region,East,North,South,West
LoyaltyLevel,,,,
Bronze,1355772.33,1505493.94,1376940.48,1560421.81
Gold,553346.17,549435.18,674290.12,633212.11
Silver,910437.72,750454.24,810709.06,808230.96


In [30]:
# Create a matrix showing each customer's purchases by product

interaction_matrix = merged_df.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

display(interaction_matrix.iloc[:5, :10])

ProductID,1,2,3,4,5,6,7,8,9,10
CustomerID,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,4,0,0,0
3,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0


In [31]:
# Find the ten most purchased products

top_10_popular_products = (
    merged_df.groupby("ProductID", as_index=False)["PurchaseCount"]
    .sum()
    .rename(columns={"PurchaseCount": "TotalPurchases"})
    .sort_values("TotalPurchases", ascending=False)
    .head(10)
    .merge(
        products_df[["ProductID", "Category", "Brand"]],
        on="ProductID",
        how="left"
    )[["ProductID", "Category", "Brand", "TotalPurchases"]]
)

display(top_10_popular_products.style.hide(axis="index"))

ProductID,Category,Brand,TotalPurchases
7,Home,SmartBuy,4923
102,Books,FitLife,3549
13,Electronics,SmartBuy,1808
134,Kitchen,SmartBuy,1361
41,Home,HomePro,1062
184,Fitness,SmartBuy,943
54,Electronics,PrimeGoods,852
195,Fitness,TechMax,801
8,Home,HomePro,686
130,Books,NextGen,577


In [32]:
# Measure how similar customers are based on their purchase history

customer_similarity = pd.DataFrame(
    cosine_similarity(interaction_matrix),
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

display(customer_similarity.iloc[:5, :5].style.format("{:.3f}"))

CustomerID,1,2,3,4,5
CustomerID,,,,,
1,1.000,0.000,0.000,0.000,0.000
2,0.000,1.000,0.000,0.000,0.000
3,0.000,0.000,1.000,0.000,0.000
4,0.000,0.000,0.000,1.000,0.000
5,0.000,0.000,0.000,0.000,1.000


In [33]:
# Recommend products using similar customers' purchase histories

def recommend_collaborative(customer_id, top_n=5):
    columns = ["ProductID", "Category", "Brand", "RecommendationScore"]

    if customer_id not in customer_similarity.index:
        return pd.DataFrame(columns=columns)

    similarities = customer_similarity.loc[customer_id].drop(customer_id)
    similarities = similarities[similarities > 0]

    if similarities.empty:
        return pd.DataFrame(columns=columns)

    scores = (
        interaction_matrix.loc[similarities.index]
        .mul(similarities, axis=0)
        .sum(axis=0)
    )

    purchased_products = interaction_matrix.loc[customer_id]
    scores = scores[purchased_products == 0]
    scores = scores[scores > 0]

    if scores.empty:
        return pd.DataFrame(columns=columns)

    recommendations = (
        scores.rename("RecommendationScore")
        .sort_values(ascending=False)
        .head(top_n)
        .rename_axis("ProductID")
        .reset_index()
        .merge(
            products_df[["ProductID", "Category", "Brand"]],
            on="ProductID",
            how="left"
        )[columns]
    )

    recommendations["RecommendationScore"] = recommendations[
        "RecommendationScore"
    ].round(3)

    return recommendations


recommend_collaborative(1)

,ProductID,Category,Brand,RecommendationScore
0,7,Home,SmartBuy,161.856
1,102,Books,FitLife,100.585
2,13,Electronics,SmartBuy,55.462
3,41,Home,HomePro,41.003
4,134,Kitchen,SmartBuy,37.728


In [34]:
# Generate collaborative-filtering recommendations for five example customers

for customer_id in [1, 2, 3, 4, 5]:
    print(f"CustomerID: {customer_id}")
    recommendations = recommend_collaborative(customer_id, top_n=5)
    display(
        recommendations.style
        .hide(axis="index")
        .format({"RecommendationScore": "{:.3f}"})
    )

CustomerID: 1


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,161.856
102,Books,FitLife,100.585
13,Electronics,SmartBuy,55.462
41,Home,HomePro,41.003
134,Kitchen,SmartBuy,37.728


CustomerID: 2


ProductID,Category,Brand,RecommendationScore
102,Books,FitLife,1189.269
13,Electronics,SmartBuy,576.900
134,Kitchen,SmartBuy,452.356
41,Home,HomePro,366.944
184,Fitness,SmartBuy,317.959


CustomerID: 3


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,134.326
102,Books,FitLife,105.682
13,Electronics,SmartBuy,62.338
134,Kitchen,SmartBuy,44.151
184,Fitness,SmartBuy,29.145


CustomerID: 4


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,686.442
13,Electronics,SmartBuy,269.228
134,Kitchen,SmartBuy,188.455
41,Home,HomePro,143.367
184,Fitness,SmartBuy,141.767


CustomerID: 5


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,102.267
102,Books,FitLife,89.314
13,Electronics,SmartBuy,41.761
134,Kitchen,SmartBuy,32.424
41,Home,HomePro,30.567


In [35]:
# Compare popularity-based and collaborative recommendations for five customers

popularity_ranking = (
    merged_df.groupby("ProductID")["PurchaseCount"]
    .sum()
    .sort_values(ascending=False)
)

comparison = []

for customer in [1, 2, 3, 4, 5]:
    purchased = set(interaction_matrix.loc[customer][
        interaction_matrix.loc[customer] > 0
    ].index)

    popularity_recommendations = [
        product_id
        for product_id in popularity_ranking.index
        if product_id not in purchased
    ][:5]

    collaborative_recommendations = recommend_collaborative(
        customer, top_n=5
    )["ProductID"].tolist()

    comparison.append({
        "CustomerID": customer,
        "PopularityBased": popularity_recommendations,
        "CollaborativeFiltering": collaborative_recommendations
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)

,CustomerID,PopularityBased,CollaborativeFiltering
0,1,"[7, 102, 13, 134, 41]","[7, 102, 13, 41, 134]"
1,2,"[102, 13, 134, 41, 184]","[102, 13, 134, 41, 184]"
2,3,"[7, 102, 13, 134, 41]","[7, 102, 13, 134, 184]"
3,4,"[7, 13, 134, 41, 184]","[7, 13, 134, 41, 184]"
4,5,"[7, 102, 13, 134, 41]","[7, 102, 13, 134, 41]"


In [ ]:
# Hold out one product for customers with at least two products

rng = np.random.default_rng(42)
eligible_customers = interaction_matrix.index[
    (interaction_matrix > 0).sum(axis=1) >= 2
].tolist()

held_out_products = {
    customer_id: rng.choice(
        interaction_matrix.loc[customer_id][
            interaction_matrix.loc[customer_id] > 0
        ].index
    )
    for customer_id in eligible_customers
}

training_matrix = interaction_matrix.copy()
for customer_id, product_id in held_out_products.items():
    training_matrix.at[customer_id, product_id] = 0

# Build recommendations from the training data

popularity_ranking_eval = training_matrix.sum(axis=0).sort_values(ascending=False)
customer_similarity_eval = pd.DataFrame(
    cosine_similarity(training_matrix),
    index=training_matrix.index,
    columns=training_matrix.index,
)

popularity_hits = 0
collaborative_hits = 0

for customer_id in eligible_customers:
    held_out_product = held_out_products[customer_id]
    purchased_products = training_matrix.loc[customer_id]
    purchased_products = purchased_products[purchased_products > 0].index

    popularity_recommendations = [
        product_id
        for product_id in popularity_ranking_eval.index
        if product_id not in purchased_products
    ][:5]
    if held_out_product in popularity_recommendations:
        popularity_hits += 1

    similarities = customer_similarity_eval.loc[customer_id].drop(customer_id)
    similarities = similarities[similarities > 0]
    if not similarities.empty:
        scores = training_matrix.loc[similarities.index].mul(
            similarities, axis=0
        ).sum(axis=0)
        collaborative_recommendations = (
            scores.drop(index=purchased_products, errors="ignore")
            .sort_values(ascending=False)
            .head(5)
            .index
            .tolist()
        )
    else:
        collaborative_recommendations = []

    if held_out_product in collaborative_recommendations:
        collaborative_hits += 1

evaluated_customers = len(eligible_customers)
evaluation_results = pd.DataFrame({
    "Method": ["Popularity-based", "Collaborative Filtering"],
    "Hits": [popularity_hits, collaborative_hits],
})
evaluation_results["HitRate@5"] = (
    evaluation_results["Hits"] / evaluated_customers
)
evaluation_results["Precision@5"] = evaluation_results["HitRate@5"] / 5
evaluation_results["Recall@5"] = evaluation_results["HitRate@5"]
display(evaluation_results.round(3))

best_method = evaluation_results.sort_values(
    ["Recall@5", "Precision@5"], ascending=False
).iloc[0]["Method"]
print(f"Best method by Recall@5: {best_method}")

,Method,Hits,HitRate@5,Precision@5,Recall@5
0,Popularity-based,204,0.210,0.042,0.210
1,Collaborative Filtering,199,0.205,0.041,0.205


Best method by Recall@5: Popularity-based


### Summery

In this task, I analysed customer, product and transaction data.

First, I cleaned the data by removing duplicates, fixing missing values and combining the datasets.

Then, I identified the customers who spend the most, the most popular products, and the categories and brands with the highest sales.

I also created recommendation methods: Popularity-Based and Collaborative Filtering.

I tested both methods to see which one recommends products better.

The results showed that Popularity-Based performed a little bit better, with a Recall@5 of 21%, compared to 20.5% for Collaborative Filtering.

As an Operations Director, this analysis helps me understand customer behaviour, identify popular products and support better stock management and product recommendations.